# Flujo de Entrenamiento YOLO

Este notebook es una plantilla limpia para entrenar, validar y probar modelos de detección.
It is organized to minimize repeated code and make experiments easier to track.


## 1. Configuración del Entorno

Ejecuta esta celda primero para cargar dependencias, detectar la raíz del proyecto y verificar CUDA.


In [2]:
from __future__ import annotations

import gc
from datetime import date
from pathlib import Path

import torch
import ultralytics
import yaml
from ultralytics import YOLO
from IPython.display import Image, display

# Resuelve la raíz del proyecto; funciona desde la raíz o desde notebooks/
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

print(f"Project root: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Torch version: {torch.__version__}")

RUN_ULTRALYTICS_CHECKS = False  # Usa True solo cuando quieras una revisión completa del entorno
if RUN_ULTRALYTICS_CHECKS:
    ultralytics.checks()


Project root: /home/robiotec/Documents/Entrenamientos/Training
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
Torch version: 2.10.0+cu128


In [2]:
print("Kernel funcionando")

Kernel funcionando


## 2. Rutas Compartidas y Valores por Defecto

Aquí se definen el modelo base y los valores generales. Normalmente no necesitas editar esta celda.


In [3]:
BASE_MODEL_PATH = PROJECT_ROOT / "model" / "yolo26n.pt"
DATA_ROOT = PROJECT_ROOT / "data"
CONFIG_ROOT = PROJECT_ROOT / "configs"

# Valores por defecto para entrenamiento. Sobrescribe por experimento solo cuando sea necesario.
DEFAULT_TRAIN_ARGS = {
    "epochs": 150,
    "imgsz": 960,
    "patience": 25,
    "device": 0,
    "warmup_epochs": 3,
    "seed": 42,
    "lrf": 0.1,
    "weight_decay": 0.0001,
    "workers": 8,
    "cache": "disk",
    "plots": True,
}

print(f"Base model: {BASE_MODEL_PATH}")
print(f"Base model exists: {BASE_MODEL_PATH.exists()}")

Base model: /home/robiotec/Documents/Entrenamientos/Training/model/yolo26n.pt
Base model exists: True


## 3. Funciones de Apoyo

Estas funciones construyen automáticamente las rutas del split, el `data.yaml`, el modelo de salida, la validación y las predicciones.


In [4]:
def clean_cuda(verbose: bool = True) -> None:
    # Libera memoria GPU en caché después de operaciones pesadas.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if verbose:
            allocated = torch.cuda.memory_allocated() / 1024**2
            reserved = torch.cuda.memory_reserved() / 1024**2
            print(f"GPU memory - allocated: {allocated:.2f} MB | reserved: {reserved:.2f} MB")


def slug(*parts: str | None) -> str:
    tokens = []
    for part in parts:
        if part is None:
            continue
        text = str(part).strip().lower().replace("-", "_").replace(" ", "_")
        text = "_".join(chunk for chunk in text.split("_") if chunk)
        if text:
            tokens.append(text)
    return "_".join(tokens)


def dataset_base_dir(mine: str, mine_code: str | None = None) -> Path:
    mine_key = mine.strip().upper()
    if mine_key == "BONANZA":
        return DATA_ROOT / "BONANZA"
    if mine_key == "TENGEL":
        if not mine_code:
            raise ValueError("TENGEL requiere mine_code, por ejemplo 'M17' o 'M93'.")
        return DATA_ROOT / "TENGEL" / mine_code.upper()
    raise ValueError(f"Mina no soportada: {mine}")


def model_output_dir(
    mine: str,
    mine_code: str | None,
    model_tag: str,
    version: str,
    run_date: str,
    aug_tag: str = "no_aug",
    model_group: str | None = None,
    model_dir_name: str | None = None,
) -> Path:
    base = dataset_base_dir(mine, mine_code) / "models"
    mine_key = mine.strip().upper()

    # BONANZA agrupa modelos históricos en aug/no_aug/only.
    # Para TENGEL, los modelos viven directo dentro de la carpeta models/ de cada metraje.
    if mine_key == "BONANZA" and model_group:
        base = base / slug(model_group)

    if model_dir_name:
        return base / model_dir_name

    if mine_key == "BONANZA":
        code = "bnz"
    else:
        code = mine_code.lower()
    model_name = slug(code, "model", model_tag, aug_tag, version, run_date)
    return base / model_name


def data_split_images_dir(data_yaml: Path, split: str = "val") -> Path:
    # Lee del YAML general tanto la raíz del dataset como la carpeta de val/test.
    with data_yaml.open("r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    if split not in config:
        raise KeyError(f"El YAML {data_yaml} no contiene la clave '{split}'.")

    dataset_root = Path(config.get("path", data_yaml.parent)).expanduser()
    if not dataset_root.is_absolute():
        dataset_root = (data_yaml.parent / dataset_root).resolve()

    images_path = Path(config[split]).expanduser()
    if images_path.is_absolute():
        return images_path
    return dataset_root / images_path


def train_model(
    run_dir: Path,
    data_yaml: Path,
    model_path: Path = BASE_MODEL_PATH,
    **overrides,
):
    args = dict(DEFAULT_TRAIN_ARGS)
    args.update(overrides)

    model = YOLO(str(model_path))
    output = model.train(
        data=str(data_yaml),
        project=str(run_dir.parent),
        name=run_dir.name,
        exist_ok=True,
        **args,
    )
    clean_cuda()
    return output


def validate_model(
    model_path: Path,
    data_yaml: Path,
    run_dir: Path,
    split: str = "val",
    conf: float = 0.25,
    **overrides,
):
    model = YOLO(str(model_path))
    output = model.val(
        data=str(data_yaml),
        split=split,
        conf=conf,
        project=str(run_dir.parent),
        name=f"{run_dir.name}__val_{split}",
        save_json=True,
        plots=True,
        exist_ok=True,
        **overrides,
    )
    clean_cuda()
    return output


def predict_images(
    model_path: Path,
    source_path: Path,
    run_dir: Path,
    conf: float = 0.25,
    **overrides,
):
    model = YOLO(str(model_path))
    output = model.predict(
        source=str(source_path),
        conf=conf,
        project=str(run_dir.parent),
        name=f"{run_dir.name}__predict",
        save=True,
        exist_ok=True,
        **overrides,
    )
    clean_cuda()
    return output

## 4. Limpieza Rápida de GPU

Run this anytime you want to free cached CUDA memory with one click.


In [5]:
clean_cuda()


GPU memory - allocated: 0.00 MB | reserved: 0.00 MB


## 5. Configurar un Entrenamiento

Para cambiar de mina o split, edita solamente esta sección. El YAML puede estar en `configs/` o dentro del propio dataset.


In [6]:
# Fecha usada en el nombre de la carpeta del modelo.
# Si quieres fijar una fecha manualmente, reemplaza `today` por "YYYY-MM-DD" en RUN_CONFIG["run_date"].
today = date.today().isoformat()

# -----------------------------------------------------------------------------
# CONFIGURACIÓN DEL ENTRENAMIENTO
# Experimento: TENGEL M93, detector de CAJA / CAJA_CACHITOS con dos clases:
#   0=caja, 1=caja_cachitos.
# Dataset: V2 depurado con VETA, BACKGROUND y MANCHA_NEGRA como negativos.
# Augmentación activa: solo flip en eje Y (mirror left-right / fliplr).
# -----------------------------------------------------------------------------
RUN_CONFIG = {
    # Mina principal para este experimento.
    "mine": "BONANZA",

    # Metraje/código de TENGEL; no aplica para BONANZA.
    "mine_code": None,

    # Etiqueta descriptiva del modelo. Solo se usa para construir el nombre de salida.
    "model_tag": "4_clases_caja_veta_cachitos",

    # Ruta relativa a PROJECT_ROOT del YAML incluido en el dataset V2.
    "data_yaml": "configs/mixto.yaml",

    # Versión lógica del modelo.
    "version": "v1",

    # Etiqueta de augmentación para el nombre del modelo.
    "aug_tag": "fliplr",

    # TENGEL guarda sus modelos directamente dentro de M93/models/.
    "model_group": None,

    # None genera un nombre representativo con mina, dataset, augmentation, versión y fecha.
    "model_dir_name": None,

    # Fecha que entra en el nombre generado si model_dir_name es None.
    "run_date": today,

    # Argumentos enviados a YOLO.train(). Se mantienen los parámetros del experimento previo.
    "train_args": {
        "imgsz": 640,
        "batch": 64,
        "epochs": 100,
        "patience": 20,
        "lr0": 0.005,

        # Color/luz apagado por ahora.
        "hsv_h": 0.0,
        "hsv_s": 0.0,
        "hsv_v": 0.0,

        # Geometría apagada salvo el flip solicitado.
        "degrees": 0.0,
        "translate": 0.0,
        "scale": 0.0,
        "shear": 0.0,
        "perspective": 0.0,

        # Flip en eje Y: espejo left-right alrededor del eje vertical de la imagen.
        "flipud": 0.0,
        "fliplr": 0.5,

        # Compuestas apagadas para aislar el efecto del flip.
        "mosaic": 0.0,
        "mixup": 0.0,
        "copy_paste": 0.0,
        "cutmix": 0.0,
        "erasing": 0.0,
        "bgr": 0.0,
        "multi_scale": False,
    },
}

# -----------------------------------------------------------------------------
# RUTAS DERIVADAS AUTOMÁTICAMENTE
# Normalmente no necesitas editar nada debajo de esta línea.
# -----------------------------------------------------------------------------
DATA_YAML = PROJECT_ROOT / RUN_CONFIG["data_yaml"]
RUN_DIR = model_output_dir(
    mine=RUN_CONFIG["mine"],
    mine_code=RUN_CONFIG["mine_code"],
    model_tag=RUN_CONFIG["model_tag"],
    version=RUN_CONFIG["version"],
    run_date=RUN_CONFIG["run_date"],
    aug_tag=RUN_CONFIG["aug_tag"],
    model_group=RUN_CONFIG["model_group"],
    model_dir_name=RUN_CONFIG["model_dir_name"],
)
MODEL_TO_VALIDATE = RUN_DIR / "weights" / "best.pt"
PREDICT_SOURCE = data_split_images_dir(DATA_YAML, split="val")

if not BASE_MODEL_PATH.exists():
    raise FileNotFoundError(f"No encuentro el modelo base: {BASE_MODEL_PATH}")
if not DATA_YAML.exists():
    raise FileNotFoundError(f"No encuentro el data.yaml/config: {DATA_YAML}")
if not PREDICT_SOURCE.exists():
    raise FileNotFoundError(f"No encuentro las imágenes de val: {PREDICT_SOURCE}")

print(f"Dataset: {RUN_CONFIG['mine']}")
print(f"Mine code: {RUN_CONFIG['mine_code']}")
print(f"Model tag: {RUN_CONFIG['model_tag']}")
print(f"Base model: {BASE_MODEL_PATH}")
print(f"Model dir: {RUN_DIR}")
print(f"YAML config: {DATA_YAML}")
print(f"Split val images: {PREDICT_SOURCE}")
print(f"Train args: {RUN_CONFIG['train_args']}")


Dataset: BONANZA
Mine code: None
Model tag: 4_clases_caja_veta_cachitos
Base model: /home/robiotec/Documents/Entrenamientos/Training/model/yolo26n.pt
Model dir: /home/robiotec/Documents/Entrenamientos/Training/data/BONANZA/models/bnz_model_4_clases_caja_veta_cachitos_fliplr_v1_2026_08_05
YAML config: /home/robiotec/Documents/Entrenamientos/Training/configs/mixto.yaml
Split val images: /home/robiotec/Documents/Entrenamientos/Training/data/BONANZA/splits/BNZ_SPLIT_4CLASS_V1/val
Train args: {'imgsz': 640, 'batch': 64, 'epochs': 100, 'patience': 20, 'lr0': 0.005, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'degrees': 0.0, 'translate': 0.0, 'scale': 0.0, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'mosaic': 0.0, 'mixup': 0.0, 'copy_paste': 0.0, 'cutmix': 0.0, 'erasing': 0.0, 'bgr': 0.0, 'multi_scale': False}


## 6. Entrenar


In [7]:

train_results = train_model(
    run_dir=RUN_DIR,
    data_yaml=DATA_YAML,
    **RUN_CONFIG["train_args"],
)


New https://pypi.org/project/ultralytics/8.4.115 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.13 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24081MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/robiotec/Documents/Entrenamientos/Training/configs/mixto.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/robiotec/Documents/En

## 7. Validar en el Split de Val

Set `MODEL_TO_VALIDATE` to your best checkpoint.

In [ ]:
if not MODEL_TO_VALIDATE.exists():
    raise FileNotFoundError(
        f"No encuentro el modelo entrenado en {MODEL_TO_VALIDATE}. Ejecuta primero la celda de entrenamiento."
    )

val_results = validate_model(
    model_path=MODEL_TO_VALIDATE,
    data_yaml=DATA_YAML,
    run_dir=RUN_DIR,
    split="val",
    conf=0.25,
    imgsz=RUN_CONFIG["train_args"]["imgsz"],
)

## 8. Predecir en Imágenes Externas (Opcional)

Set `PREDICT_SOURCE` to any folder of images.


In [ ]:
print(f"Prediction source: {PREDICT_SOURCE}")
print(f"Prediction source exists: {PREDICT_SOURCE.exists()}")

# Predicción visual opcional sobre val. Puede tardar porque el val completo tiene varias centenas de imágenes.
# Descomenta este bloque cuando ya tengas el entrenamiento validado.
# pred_results = predict_images(
#     model_path=MODEL_TO_VALIDATE,
#     source_path=PREDICT_SOURCE,
#     run_dir=RUN_DIR,
#     conf=0.25,
#     imgsz=RUN_CONFIG["train_args"]["imgsz"],
# )

## 9. Comparación Visual Rápida (Opcional)


In [ ]:

# Actualiza estas rutas si quieres comparar matrices de confusión lado a lado.
# from IPython.display import Image, display
#
# train_img = Image(filename=str(RUN_DIR / "confusion_matrix_normalized.png"), width=450)
# val_img = Image(filename=str(RUN_DIR.parent / f"{RUN_DIR.name}__val_test" / "confusion_matrix_normalized.png"), width=450)
#
# try:
#     from ipywidgets import HBox
#     display(HBox([train_img, val_img]))
# except Exception:
#     display(train_img)
#     display(val_img)
